# 01 · The frozen feature bank

Fine-tuning a 300M-parameter backbone takes hours per experiment on a laptop. Running each
backbone over the corpus **once** and caching the pooled embeddings turns every downstream
question — head architecture, calibration, conformal sets, abstention, ensembling — into
something that finishes in seconds.

That one decision is why this project could afford to be rigorous: the ablation in notebook
02 re-ran dozens of times against this cache. Without it, each of those runs would have
been a fresh forward pass over 101,000 images.

In [1]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REPORTS = ROOT / "artifacts" / "reports"

# Chart palette, validated for the lightness band, chroma floor, colour-vision
# separation and contrast. The interface accents fail as data marks: they sit
# too light and drop under 3:1 against the page. Order matters — green beside
# amber fails deuteranope separation, so teal sits between them.
INK, DIM = "#0b0b0f", "#4b4b55"
C = ["#e62429", "#1b4ce0", "#c07708", "#0e8fa3", "#16a34a"]

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 120,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": DIM, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": DIM, "ytick.color": DIM,
    "axes.grid": True, "grid.color": "#e6e4dc", "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
    "legend.frameon": False,
})

## Measured throughput

Benchmarked on an M3 Pro, batch 8, fp16, before committing to the extraction.

In [2]:
bench = pd.DataFrame([
    ("SigLIP-SO400M", "vit_so400m_patch14_siglip_384.webli", 384, 428.2, 5.3),
    ("EVA-02-L",      "eva02_large_patch14_448...in1k",      448, 304.1, 4.1),
    ("DINOv2-L",      "vit_large_patch14_dinov2.lvd142m",    518, 304.4, 3.6),
], columns=["backbone", "timm name", "px", "params (M)", "img/s"])
bench["hours for 101k"] = (101_000 / bench["img/s"] / 3600).round(1)
bench

,backbone,timm name,px,params (M),img/s,hours for 101k
0,SigLIP-SO400M,vit_so400m_patch14_siglip_384.webli,384,428.2,5.3,5.3
1,EVA-02-L,eva02_large_patch14_448...in1k,448,304.1,4.1,6.8
2,DINOv2-L,vit_large_patch14_dinov2.lvd142m,518,304.4,3.6,7.8


Roughly twenty hours in total, run sequentially in the background. A one-time cost against
an unbounded number of cheap experiments afterwards.

## What actually landed

In [3]:
FEATURES = ROOT / "artifacts" / "features"
rows = []
for d in sorted(FEATURES.iterdir()):
    if not d.is_dir():
        continue
    for split in ("train", "test"):
        meta_path = d / f"{split}_meta.json"
        if meta_path.exists():
            m = json.loads(meta_path.read_text())
            rows.append({
                "backbone": m["backbone"], "split": m["split"], "images": m["count"],
                "dim": m["dim"], "px": m["image_size"],
                "img/s": round(m["images_per_second"], 2),
                "hours": round(m["seconds"] / 3600, 2),
            })
pd.DataFrame(rows)

,backbone,split,images,dim,px,img/s,hours
0,dinov2_large,train,75750,1024,518,3.36,6.26
1,dinov2_large,test,25250,1024,518,1.92,3.66
2,eva02_large,train,75750,1024,448,4.04,5.21
3,eva02_large,test,25250,1024,448,3.87,1.81
4,siglip_so400m,train,75750,1152,384,5.16,4.08
5,siglip_so400m,test,25250,1152,384,5.22,1.34


## The split, which everything else depends on

Food-101 ships only *train* and *test*. Reporting model-selection numbers on test is the
single most common way projects like this overstate themselves, so a fixed 4%
class-stratified slice is carved out of **train** and shared by every pipeline. The test
split is untouched until final evaluation.

The RNG is shared across the class loop rather than re-seeded per class, so the result
depends on class order as well as the seed — which is why the mask is hashed and checked
rather than assumed to match between machines.

In [4]:
import sys
sys.path.insert(0, str(ROOT / "src"))
import hashlib
from nutrivision.data.dataset import holdout_mask

y = np.load(FEATURES / "siglip_so400m" / "train_y.npy")
mask = holdout_mask(y, 101, 0.04, 1337)

print(f"train images     {len(y):,}")
print(f"per class        {np.bincount(y).min()}–{np.bincount(y).max()}")
print(f"validation       {mask.sum():,}")
print(f"training         {(~mask).sum():,}")
print(f"mask sha256      {hashlib.sha256(mask.tobytes()).hexdigest()}")

train images     75,750
per class        750–750
validation       3,030
training         72,720
mask sha256      a1e99550d007a01ab5654f2125816155307ba1b27d7329cd23cf4a3bcfa170d7


That hash is asserted inside the Kaggle fine-tuning notebook before a single step of
training runs. A different slice there would make the fine-tune incomparable to everything
measured here, and would leak validation data into any ensemble of the two.